# Experiment Tracking — MLflow (Phase 1 / MLOps)

Every model run in Pipelines 1 and 2 logs its parameters, per-fold-aggregated metrics (mean/std of R2, RMSE, NRMSE, MAE), the generalization gap, and the raw fold table as an artifact to a local SQLite-backed MLflow tracking store (`mlflow.db` at the repo root). This notebook queries that store directly (no retraining) and summarizes every experiment and run.

To browse runs interactively (with the plots/artifacts UI), launch:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

All summary tables produced here are saved to `results/mlflow/`, and the last cell shows the complete combined results.

Setup: point MLflow at the local SQLite store and configure pandas to display full tables.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)

from src.config import PROJECT_ROOT, get_path
from src.utils.io import save_table

db_path = PROJECT_ROOT / 'mlflow.db'
mlflow.set_tracking_uri(f'sqlite:///{db_path.as_posix()}')
client = MlflowClient()
print('Tracking URI:', mlflow.get_tracking_uri())

Tracking URI: sqlite:///C:/Users/mohammadhosein/Desktop/DeepLearning-miniProject/mlflow.db


## Experiments overview

Every experiment corresponds to one `pipeline{1,2}_<dataset>` run (Pipelines 1 and 2 log to MLflow; Pipelines 3/4, tuning and XAI do not).

In [2]:
experiments = client.search_experiments()
overview = pd.DataFrame(
    [
        {
            'experiment_id': e.experiment_id,
            'experiment_name': e.name,
            'n_runs': len(client.search_runs([e.experiment_id])),
        }
        for e in experiments
    ]
).sort_values('experiment_name').reset_index(drop=True)
overview

,experiment_id,experiment_name,n_runs
0,0,Default,0
1,1,pipeline1_Dataset_0136,154
2,2,pipeline1_Dataset_0172,110
3,3,pipeline1_Dataset_3772,110
4,4,pipeline2_Dataset_0136,22
5,5,pipeline2_Dataset_0172,22
6,6,pipeline2_Dataset_3772,22
7,7,pipeline3_Dataset_0136,42
8,8,pipeline3_Dataset_0172,30
9,9,pipeline3_Dataset_3772,30


## All runs, every experiment

`mlflow.search_runs` returns one row per run with every logged parameter and metric as a column.

In [3]:
all_runs = mlflow.search_runs(experiment_ids=[e.experiment_id for e in experiments], search_all_experiments=True)
print('Total logged runs:', len(all_runs))
all_runs

Total logged runs: 560


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.train_mae_std,metrics.test_r2_std,metrics.gap_r2,metrics.train_rmse_std,metrics.test_rmse_mean,metrics.test_mae_mean,metrics.train_rmse_mean,metrics.train_nrmse_mean,metrics.train_mae_mean,metrics.gap_rmse,metrics.train_r2_std,metrics.test_nrmse_std,metrics.train_r2_mean,metrics.test_nrmse_mean,metrics.test_rmse_std,metrics.test_r2_mean,metrics.train_nrmse_std,metrics.test_mae_std,params.discrete,params.model,params.mode,params.targets,params.dataset,params.target,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.name
0,b0f9688e3e4e486b992e71ea7e496b1b,12,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:54.434000+00:00,2026-07-18 21:44:54.920000+00:00,0.086508,0.194615,0.219680,0.363230,3.661759,1.748661,1.787715,0.009819,0.810085,1.874045,0.008175,0.007899,0.958450,0.021177,1.295298,0.738770,0.002057,0.431267,True,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,NOTEBOOK,lightgbm__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
1,7cb3131e463848eda1693828d843619a,12,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:48.425000+00:00,2026-07-18 21:44:48.828000+00:00,0.148067,0.107826,0.209728,0.302534,3.494039,1.772996,1.505226,0.008270,0.777055,1.988814,0.013352,0.005645,0.958615,0.020188,0.910243,0.748887,0.001733,0.381751,True,gpr,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,NOTEBOOK,gpr__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
2,72a33522e6ba46e48418e88d675b59ba,12,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:45.947000+00:00,2026-07-18 21:44:46.418000+00:00,0.074792,0.155423,0.197856,0.232943,3.464469,1.689501,1.427529,0.007841,0.701445,2.036940,0.004768,0.006268,0.970795,0.020036,1.017304,0.772938,0.001344,0.394142,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,NOTEBOOK,xgboost__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
3,c788b3da1524435aaa3f540685e6dad8,11,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:34.296000+00:00,2026-07-18 21:44:34.739000+00:00,0.053485,0.026448,0.068730,0.143667,2.631003,1.277764,1.023234,0.005612,0.520181,1.607769,0.002210,0.003451,0.984865,0.015161,0.622997,0.916135,0.000791,0.250446,True,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,NOTEBOOK,lightgbm__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
4,ce3c3e07e9cf4e9d9baa0492551dec61,11,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:27.236000+00:00,2026-07-18 21:44:27.656000+00:00,0.081697,0.044612,0.113534,0.170365,2.888986,1.490980,1.337294,0.007339,0.704073,1.551692,0.006423,0.003146,0.969027,0.016686,0.545157,0.855493,0.000963,0.208056,True,gpr,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,NOTEBOOK,gpr__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
5,d569127de58446e6b094f0c72cefd8c4,11,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:23.988000+00:00,2026-07-18 21:44:24.476000+00:00,0.054673,0.024416,0.058853,0.160005,2.648487,1.231394,1.454139,0.007980,0.677201,1.194348,0.003122,0.003563,0.977683,0.015247,0.647393,0.918830,0.000911,0.215759,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,NOTEBOOK,xgboost__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
6,c3247d41699340b7957cdc29ff32d415,10,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:12.006000+00:00,2026-07-18 21:44:12.427000+00:00,0.067925,0.040665,0.070091,0.119463,2.332151,1.387738,0.999666,0.002336,0.573578,1.332485,0.002575,0.001186,0.986940,0.005690,0.507603,0.916849,0.000279,0.248628,Tru

## Best run per experiment (by mean test R2)

In [4]:
if 'metrics.test_r2_mean' in all_runs.columns:
    best_per_experiment = (
        all_runs.sort_values('metrics.test_r2_mean', ascending=False)
        .groupby('experiment_id', as_index=False)
        .first()
    )
    key_cols = [
        c for c in best_per_experiment.columns
        if c in ('experiment_id', 'run_id', 'tags.mlflow.runName', 'status')
        or c.startswith('params.') or c.startswith('metrics.')
    ]
    best_per_experiment = best_per_experiment.merge(
        overview[['experiment_id', 'experiment_name']], on='experiment_id'
    )
    display_cols = ['experiment_name'] + [c for c in key_cols if c != 'experiment_id']
    display(best_per_experiment[display_cols])
else:
    print('No runs with metrics.test_r2_mean found yet — run notebook 02 first.')
    best_per_experiment = pd.DataFrame()
    display_cols = []

,experiment_name,run_id,status,metrics.train_mae_std,metrics.test_r2_std,metrics.gap_r2,metrics.train_rmse_std,metrics.test_rmse_mean,metrics.test_mae_mean,metrics.train_rmse_mean,metrics.train_nrmse_mean,metrics.train_mae_mean,metrics.gap_rmse,metrics.train_r2_std,metrics.test_nrmse_std,metrics.train_r2_mean,metrics.test_nrmse_mean,metrics.test_rmse_std,metrics.test_r2_mean,metrics.train_nrmse_std,metrics.test_mae_std,params.discrete,params.model,params.mode,params.targets,params.dataset,params.target,tags.mlflow.runName
0,pipeline1_Dataset_0136,4eb3fadfe0fa466299eb959f67642603,FINISHED,0.091892,0.002644,0.003927,0.099648,1.470655,1.195909,0.613733,0.006428,0.486941,0.856922,0.000206,0.007028,0.999405,0.022331,0.368590,0.995478,0.001140,0.316250,False,gpr,single_output,None,Dataset_0136,Strain,gpr__Strain
1,pipeline4_Dataset_0136,c3247d41699340b7957cdc29ff32d415,FINISHED,0.067925,0.040665,0.070091,0.119463,2.332151,1.387738,0.999666,0.002336,0.573578,1.332485,0.002575,0.001186,0.986940,0.005690,0.507603,0.916849,0.000279,0.248628,True,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0136,None,lightgbm__multi
2,pipeline4_Dataset_0172,d569127de58446e6b094f0c72cefd8c4,FINISHED,0.054673,0.024416,0.058853,0.160005,2.648487,1.231394,1.454139,0.007980,0.677201,1.194348,0.003122,0.003563,0.977683,0.015247,0.647393,0.918830,0.000911,0.215759,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,xgboost__multi
3,pipeline4_Dataset_3772,80dcf225f9ee4008b9c43407603a8166,FINISHED,0.074792,0.155423,0.197856,0.232943,3.464469,1.689501,1.427529,0.007841,0.701445,2.036940,0.004768,0.006268,0.970795,0.020036,1.017304,0.772938,0.001344,0.394142,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,xgboost__multi
4,pipeline1_Dataset_0172,92223c0543ad4087b0d34681d9be7ab6,FINISHED,0.005464,0.005073,0.007733,0.005942,0.146739,0.110221,0.089248,0.015201,0.067008,0.057492,0.000442,0.006172,0.996152,0.030715,0.041609,0.988418,0.001120,0.024934,False,xgboost,single_output,None,Dataset_0172,wear rate,xgboost__wear rate
5,pipeline1_Dataset_3772,043574f85f604a0488fe6015674f1680,FINISHED,0.002621,0.050311,0.057816,0.003097,0.050198,0.039668,0.026183,0.034563,0.019312,0.024015,0.004513,0.032863,0.983774,0.090063,0.025224,0.925959,0.005963,0.018491,False,xgboost,single_output,None,Dataset_3772,Si Particle Size (um),xgboost__Si Particle Size (um)
6,pipeline2_Dataset_0136,98eacf5ad3d44f67ba6d4c4c9aca9a09,FINISHED,0.067925,0.040665,0.070091,0.119463,2.332151,1.387738,0.999666,0.002336,0.573578,1.332485,0.002575,0.001186,0.986940,0.005690,0.507603,0.916849,0.000279,0.248628,False,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0136,None,lightgbm__multi
7,pipeline2_Dataset_0172,97ae7299bf2e4730b20940be813e9386,FINISHED,0.054673,0.024416,0.058853,0.160005,2.648487,1.231394,1.454139,0.007980,0.677201,1.194348,0.003122,0.003563,0.977683,0.015247,0.647393,0.918830,0.000911,0.215759,False,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,xgboost__multi
8,pipeline2_Dataset_3772,636cf48830fd4ed09650030aa7df6a13,FINISHED,0.074792,0.155423,0.197856,0.232943,3.464469,1.689501,1.427529,0.007841,0.701445,2.036940,0.004768,0.006268,0.970795,0.020036,1.017304,0.772938,0.001344,0.394142,False,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,xgboost__multi
9,pipeline3_Dataset_0136,153223baea114afdb58a9171747082af,FINISHED,0.107212,0.004269,0.007662,0.122478,2.520594,2.156492,0.904912,0.007748,0.726056,1.615682,0.000319,0.009012,0.999068,0.030272,0.734421,0.991406,0.001217,0.642343,True,xgboost,single_output,None,Dataset_0136,Temperature (°C),xgboost__Temperature (°C)


## Top 15 runs overall (by mean test R2)

In [5]:
if not all_runs.empty and 'metrics.test_r2_mean' in all_runs.columns:
    metric_cols = [c for c in all_runs.columns if c.startswith('metrics.')]
    param_cols = [c for c in all_runs.columns if c.startswith('params.')]
    top_runs = all_runs.sort_values('metrics.test_r2_mean', ascending=False).head(15)
    display(top_runs[['experiment_id', 'tags.mlflow.runName'] + param_cols + metric_cols])
else:
    top_runs = pd.DataFrame()
    print('No runs available yet.')

,experiment_id,tags.mlflow.runName,params.discrete,params.model,params.mode,params.targets,params.dataset,params.target,metrics.train_mae_std,metrics.test_r2_std,metrics.gap_r2,metrics.train_rmse_std,metrics.test_rmse_mean,metrics.test_mae_mean,metrics.train_rmse_mean,metrics.train_nrmse_mean,metrics.train_mae_mean,metrics.gap_rmse,metrics.train_r2_std,metrics.test_nrmse_std,metrics.train_r2_mean,metrics.test_nrmse_mean,metrics.test_rmse_std,metrics.test_r2_mean,metrics.train_nrmse_std,metrics.test_mae_std
209,1,gpr__Strain,False,gpr,single_output,None,Dataset_0136,Strain,0.091892,0.002644,0.003927,0.099648,1.470655,1.195909,0.613733,0.006428,0.486941,0.856922,0.000206,0.007028,0.999405,0.022331,0.368590,0.995478,0.001140,0.316250
489,1,gpr__Strain,False,gpr,single_output,None,Dataset_0136,Strain,0.091892,0.002644,0.003927,0.099648,1.470655,1.195909,0.613733,0.006428,0.486941,0.856922,0.000206,0.007028,0.999405,0.022331,0.368590,0.995478,0.001140,0.316250
324,7,xgboost__Temperature (°C),True,xgboost,single_output,None,Dataset_0136,Temperature (°C),0.107212,0.004269,0.007662,0.122478,2.520594,2.156492,0.904912,0.007748,0.726056,1.615682,0.000319,0.009012,0.999068,0.030272,0.734421,0.991406,0.001217,0.642343
216,1,xgboost__Temperature (°C),False,xgboost,single_output,None,Dataset_0136,Temperature (°C),0.107212,0.004269,0.007662,0.122478,2.520594,2.156492,0.904912,0.007748,0.726056,1.615682,0.000319,0.009012,0.999068,0.030272,0.734421,0.991406,0.001217,0.642343
44,7,xgboost__Temperature (°C),True,xgboost,single_output,None,Dataset_0136,Temperature (°C),0.107212,0.004269,0.007662,0.122478,2.520594,2.156492,0.904912,0.007748,0.726056,1.615682,0.000319,0.009012,0.999068,0.030272,0.734421,0.991406,0.001217,0.642343
496,1,xgboost__Temperature (°C),False,xgboost,single_output,None,Dataset_0136,Temperature (°C),0.107212,0.004269,0.007662,0.122478,2.520594,2.156492,0.904912,0.007748,0.726056,1.615682,0.000319,0.009012,0.999068,0.030272,0.734421,0.991406,0.001217,0.642343
500,1,gpr__Temperature (°C),False,gpr,single_output,None,Dataset_0136,Temperature (°C),0.183845,0.006602,0.007533,0.237432,2.433059,1.919989,1.121643,0.009576,0.845655,1.311416,0.000589,0.011382,0.998552,0.029514,0.843652,0.991019,0.002075,0.722392
220,1,gpr__Temperature (°C),False,gpr,single_output,None,Dataset_0136,Temperature (°C),0.183845,0.006602,0.007533,0.237432,2.433059,1.919989,1.121643,0.009576,0.845655,1.311416,0.000589,0.011382,0.998552,0.029514,0.843652,0.991019,0.002075,0.722392
205,1,xgboost__Strain,False,xgboost,single_output,None,Dataset_0136,Strain,0.111393,0.007270,0.008726,0.148457,2.100767,1.763716,0.741377,0.007762,0.603808,1.359389,0.000429,0.011174,0.999116,0.031570,0.734428,0.990390,0.001633,0.599267
41,7,xgboost__Strain,True,xgboost,single_output,None,Dataset_0136,Strain,0.111393,0.007270,0.008726,0.148457,2.100767,1.763716,0.741377,0.007762,0.603808,1.359389,0.000429,0.011174,0.999116,0.031570,0.734428,0.990390,0.001633,0.599267


## Final Summary — Complete MLflow Tracking Overview

Experiments overview, every run, and the best run per experiment, saved to `results/mlflow/`.

In [6]:
mlflow_out_dir = get_path('results_dir') / 'mlflow'
save_table(overview, mlflow_out_dir / 'experiments_overview.csv')
save_table(all_runs, mlflow_out_dir / 'all_runs.csv')
if not best_per_experiment.empty:
    save_table(best_per_experiment, mlflow_out_dir / 'best_run_per_experiment.csv')

print('=' * 90)
print('1) EXPERIMENTS OVERVIEW')
print('=' * 90)
display(overview)

print()
print('=' * 90)
print(f'2) ALL {len(all_runs)} LOGGED RUNS')
print('=' * 90)
display(all_runs)

if not best_per_experiment.empty:
    print()
    print('=' * 90)
    print('3) BEST RUN PER EXPERIMENT (highest mean test R2)')
    print('=' * 90)
    display(best_per_experiment[display_cols])

print()
print('MLflow tracking store:', db_path.resolve())
print('Launch the UI with: mlflow ui --backend-store-uri sqlite:///mlflow.db')
print('Summary tables saved to:', mlflow_out_dir.resolve())

1) EXPERIMENTS OVERVIEW


,experiment_id,experiment_name,n_runs
0,0,Default,0
1,1,pipeline1_Dataset_0136,154
2,2,pipeline1_Dataset_0172,110
3,3,pipeline1_Dataset_3772,110
4,4,pipeline2_Dataset_0136,22
5,5,pipeline2_Dataset_0172,22
6,6,pipeline2_Dataset_3772,22
7,7,pipeline3_Dataset_0136,42
8,8,pipeline3_Dataset_0172,30
9,9,pipeline3_Dataset_3772,30



2) ALL 560 LOGGED RUNS


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.train_mae_std,metrics.test_r2_std,metrics.gap_r2,metrics.train_rmse_std,metrics.test_rmse_mean,metrics.test_mae_mean,metrics.train_rmse_mean,metrics.train_nrmse_mean,metrics.train_mae_mean,metrics.gap_rmse,metrics.train_r2_std,metrics.test_nrmse_std,metrics.train_r2_mean,metrics.test_nrmse_mean,metrics.test_rmse_std,metrics.test_r2_mean,metrics.train_nrmse_std,metrics.test_mae_std,params.discrete,params.model,params.mode,params.targets,params.dataset,params.target,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.name
0,b0f9688e3e4e486b992e71ea7e496b1b,12,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:54.434000+00:00,2026-07-18 21:44:54.920000+00:00,0.086508,0.194615,0.219680,0.363230,3.661759,1.748661,1.787715,0.009819,0.810085,1.874045,0.008175,0.007899,0.958450,0.021177,1.295298,0.738770,0.002057,0.431267,True,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,NOTEBOOK,lightgbm__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
1,7cb3131e463848eda1693828d843619a,12,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:48.425000+00:00,2026-07-18 21:44:48.828000+00:00,0.148067,0.107826,0.209728,0.302534,3.494039,1.772996,1.505226,0.008270,0.777055,1.988814,0.013352,0.005645,0.958615,0.020188,0.910243,0.748887,0.001733,0.381751,True,gpr,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,NOTEBOOK,gpr__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
2,72a33522e6ba46e48418e88d675b59ba,12,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:45.947000+00:00,2026-07-18 21:44:46.418000+00:00,0.074792,0.155423,0.197856,0.232943,3.464469,1.689501,1.427529,0.007841,0.701445,2.036940,0.004768,0.006268,0.970795,0.020036,1.017304,0.772938,0.001344,0.394142,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,NOTEBOOK,xgboost__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
3,c788b3da1524435aaa3f540685e6dad8,11,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:34.296000+00:00,2026-07-18 21:44:34.739000+00:00,0.053485,0.026448,0.068730,0.143667,2.631003,1.277764,1.023234,0.005612,0.520181,1.607769,0.002210,0.003451,0.984865,0.015161,0.622997,0.916135,0.000791,0.250446,True,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,NOTEBOOK,lightgbm__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
4,ce3c3e07e9cf4e9d9baa0492551dec61,11,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:27.236000+00:00,2026-07-18 21:44:27.656000+00:00,0.081697,0.044612,0.113534,0.170365,2.888986,1.490980,1.337294,0.007339,0.704073,1.551692,0.006423,0.003146,0.969027,0.016686,0.545157,0.855493,0.000963,0.208056,True,gpr,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,NOTEBOOK,gpr__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
5,d569127de58446e6b094f0c72cefd8c4,11,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:23.988000+00:00,2026-07-18 21:44:24.476000+00:00,0.054673,0.024416,0.058853,0.160005,2.648487,1.231394,1.454139,0.007980,0.677201,1.194348,0.003122,0.003563,0.977683,0.015247,0.647393,0.918830,0.000911,0.215759,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,NOTEBOOK,xgboost__multi,mohammadhosein,C:\Users\mohammadhosein\AppData\Local\Programs...
6,c3247d41699340b7957cdc29ff32d415,10,FINISHED,file:///C:/Users/mohammadhosein/Desktop/surrog...,2026-07-18 21:44:12.006000+00:00,2026-07-18 21:44:12.427000+00:00,0.067925,0.040665,0.070091,0.119463,2.332151,1.387738,0.999666,0.002336,0.573578,1.332485,0.002575,0.001186,0.986940,0.005690,0.507603,0.916849,0.000279,0.248628,Tru


3) BEST RUN PER EXPERIMENT (highest mean test R2)


,experiment_name,run_id,status,metrics.train_mae_std,metrics.test_r2_std,metrics.gap_r2,metrics.train_rmse_std,metrics.test_rmse_mean,metrics.test_mae_mean,metrics.train_rmse_mean,metrics.train_nrmse_mean,metrics.train_mae_mean,metrics.gap_rmse,metrics.train_r2_std,metrics.test_nrmse_std,metrics.train_r2_mean,metrics.test_nrmse_mean,metrics.test_rmse_std,metrics.test_r2_mean,metrics.train_nrmse_std,metrics.test_mae_std,params.discrete,params.model,params.mode,params.targets,params.dataset,params.target,tags.mlflow.runName
0,pipeline1_Dataset_0136,4eb3fadfe0fa466299eb959f67642603,FINISHED,0.091892,0.002644,0.003927,0.099648,1.470655,1.195909,0.613733,0.006428,0.486941,0.856922,0.000206,0.007028,0.999405,0.022331,0.368590,0.995478,0.001140,0.316250,False,gpr,single_output,None,Dataset_0136,Strain,gpr__Strain
1,pipeline4_Dataset_0136,c3247d41699340b7957cdc29ff32d415,FINISHED,0.067925,0.040665,0.070091,0.119463,2.332151,1.387738,0.999666,0.002336,0.573578,1.332485,0.002575,0.001186,0.986940,0.005690,0.507603,0.916849,0.000279,0.248628,True,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0136,None,lightgbm__multi
2,pipeline4_Dataset_0172,d569127de58446e6b094f0c72cefd8c4,FINISHED,0.054673,0.024416,0.058853,0.160005,2.648487,1.231394,1.454139,0.007980,0.677201,1.194348,0.003122,0.003563,0.977683,0.015247,0.647393,0.918830,0.000911,0.215759,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,xgboost__multi
3,pipeline4_Dataset_3772,80dcf225f9ee4008b9c43407603a8166,FINISHED,0.074792,0.155423,0.197856,0.232943,3.464469,1.689501,1.427529,0.007841,0.701445,2.036940,0.004768,0.006268,0.970795,0.020036,1.017304,0.772938,0.001344,0.394142,True,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,xgboost__multi
4,pipeline1_Dataset_0172,92223c0543ad4087b0d34681d9be7ab6,FINISHED,0.005464,0.005073,0.007733,0.005942,0.146739,0.110221,0.089248,0.015201,0.067008,0.057492,0.000442,0.006172,0.996152,0.030715,0.041609,0.988418,0.001120,0.024934,False,xgboost,single_output,None,Dataset_0172,wear rate,xgboost__wear rate
5,pipeline1_Dataset_3772,043574f85f604a0488fe6015674f1680,FINISHED,0.002621,0.050311,0.057816,0.003097,0.050198,0.039668,0.026183,0.034563,0.019312,0.024015,0.004513,0.032863,0.983774,0.090063,0.025224,0.925959,0.005963,0.018491,False,xgboost,single_output,None,Dataset_3772,Si Particle Size (um),xgboost__Si Particle Size (um)
6,pipeline2_Dataset_0136,98eacf5ad3d44f67ba6d4c4c9aca9a09,FINISHED,0.067925,0.040665,0.070091,0.119463,2.332151,1.387738,0.999666,0.002336,0.573578,1.332485,0.002575,0.001186,0.986940,0.005690,0.507603,0.916849,0.000279,0.248628,False,lightgbm,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0136,None,lightgbm__multi
7,pipeline2_Dataset_0172,97ae7299bf2e4730b20940be813e9386,FINISHED,0.054673,0.024416,0.058853,0.160005,2.648487,1.231394,1.454139,0.007980,0.677201,1.194348,0.003122,0.003563,0.977683,0.015247,0.647393,0.918830,0.000911,0.215759,False,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_0172,None,xgboost__multi
8,pipeline2_Dataset_3772,636cf48830fd4ed09650030aa7df6a13,FINISHED,0.074792,0.155423,0.197856,0.232943,3.464469,1.689501,1.427529,0.007841,0.701445,2.036940,0.004768,0.006268,0.970795,0.020036,1.017304,0.772938,0.001344,0.394142,False,xgboost,multi_output,"Si Particle Size (um),Hardness (HV),wear rate,...",Dataset_3772,None,xgboost__multi
9,pipeline3_Dataset_0136,153223baea114afdb58a9171747082af,FINISHED,0.107212,0.004269,0.007662,0.122478,2.520594,2.156492,0.904912,0.007748,0.726056,1.615682,0.000319,0.009012,0.999068,0.030272,0.734421,0.991406,0.001217,0.642343,True,xgboost,single_output,None,Dataset_0136,Temperature (°C),xgboost__Temperature (°C)



MLflow tracking store: C:\Users\mohammadhosein\Desktop\DeepLearning-miniProject\mlflow.db
Launch the UI with: mlflow ui --backend-store-uri sqlite:///mlflow.db
Summary tables saved to: C:\Users\mohammadhosein\Desktop\DeepLearning-miniProject\results\mlflow
